In [1]:
import pandas as pd

df = pd.read_csv("fra_cleaned.csv", sep=";", encoding="cp1252", decimal=",")
df.shape

(24063, 18)

In [2]:
df.head()

,url,Perfume,Brand,Country,Gender,Rating Value,Rating Count,Year,Top,Middle,Base,Perfumer1,Perfumer2,mainaccord1,mainaccord2,mainaccord3,mainaccord4,mainaccord5
0,https://www.fragrantica.com/perfume/xerjoff/ac...,accento-overdose-pride-edition,xerjoff,Italy,unisex,1.42,201,2022.0,"fruity notes, aldehydes, green notes","bulgarian rose, egyptian jasmine, lily-of-the-...","eucalyptus, pine",unknown,NaN,rose,woody,fruity,aromatic,floral
1,https://www.fragrantica.com/perfume/jean-paul-...,classique-pride-2024,jean-paul-gaultier,France,women,1.86,70,2024.0,"yuzu, citruses","orange blossom, neroli","musk, blonde woods",unknown,NaN,citrus,white floral,sweet,fresh,musky
2,https://www.fragrantica.com/perfume/jean-paul-...,classique-pride-2023,jean-paul-gaultier,France,unisex,1.91,285,2023.0,"blood orange, yuzu","neroli, orange blossom","musk, white woods",natalie gracia-cetto,quentin bisch,citrus,white floral,sweet,fresh spicy,musky
3,https://www.fragrantica.com/perfume/bruno-bana...,pride-edition-man,bruno-banani,Germany,men,1.92,59,2019.0,"guarana, grapefruit, red apple","walnut, lavender, guava","vetiver, benzoin, amber",unknown,NaN,fruity,nutty,woody,tropical,NaN
4,https://www.fragrantica.com/perfume/jean-paul-...,le-male-pride-collector,jean-paul-gaultier,France,men,1.93,632,2020.0,"mint, lavender, cardamom, artemisia, bergamot","caraway, cinnamon, orange blossom","vanilla, sandalwood, amber, cedar, tonka bean",francis kurkdjian,NaN,aromatic,warm spicy,fresh spicy,cinnamon,vanilla


In [3]:
# Geographic definition: New World = Americas + Oceania; Old World = Europe, Asia, Africa
NEW_WORLD = {"USA", "Canada", "Mexico", "Brazil", "Argentina", "Peru", "Bermuda", "Australia"}

df["new_world"] = df["Country"].isin(NEW_WORLD).astype(int)
df["new_world"].value_counts()

new_world
0    18027
1     6036
Name: count, dtype: int64

In [4]:
# Global South: political/historical grouping, not income — Gulf states included; Russia & Israel counted as North
GLOBAL_SOUTH = {
    "Brazil", "Argentina", "Peru", "Mexico",
    "UAE", "Arabia saudi", "Oman", "Bahrain", "Bahrein", "Kuwait", "Qatar",
    "Turkey", "Lebanon", "Egypt", "Morocco", "Iran", "Pakistan", "India",
    "Thailand", "Indonesia", "Malaysia", "China", "Hong Kong", "Kazakhstan",
}

df["global_south"] = df["Country"].isin(GLOBAL_SOUTH).astype(int)
pd.crosstab(df["global_south"], df["new_world"])

new_world,0,1
global_south,,
0,16282,4606
1,1745,1430


In [5]:
import numpy as np

# Spelling variants and synonyms -> one canonical note name (see note_aliases.csv)
NOTE_ALIASES = pd.read_csv("note_aliases.csv").set_index("variant")["canonical"].to_dict()

# Brand-owned house blends listed as notes: they identify one brand, not an ingredient, so they're excluded
HOUSE_BLENDS = {"accord eudora"}

def clean(name):
    """Strip whitespace and trademark symbols (e.g. "orcanox™" -> "orcanox")."""
    return name.replace("®", "").replace("™", "").strip()

# Multi-hot feature matrices: one 0/1 column per note/accord, presence per fragrance
def multi_hot(series_list, aliases={}):
    items = pd.concat(series_list, axis=1).apply(
        lambda row: {aliases.get(clean(x), clean(x)) for cell in row.dropna() for x in str(cell).split(",")
                     if clean(x) and clean(x) not in HOUSE_BLENDS},
        axis=1,
    )
    return pd.Series(items).str.join("|").str.get_dummies()

features = {
    "top": multi_hot([df["Top"]], NOTE_ALIASES),
    "middle": multi_hot([df["Middle"]], NOTE_ALIASES),
    "base": multi_hot([df["Base"]], NOTE_ALIASES),
    "notes": multi_hot([df["Top"], df["Middle"], df["Base"]], NOTE_ALIASES),  # any position, counted once per fragrance
    "accords": multi_hot([df[f"mainaccord{i}"] for i in range(1, 6)]),
}

# Drop rare items once, on the full dataset, so every comparison uses the same items
MIN_COUNT = 10
features = {k: X.loc[:, X.sum() >= MIN_COUNT] for k, X in features.items()}
{k: v.shape for k, v in features.items()}

{'top': (24063, 404),
 'middle': (24063, 459),
 'base': (24063, 286),
 'notes': (24063, 712),
 'accords': (24063, 67)}

In [6]:
# Gender: women vs men (unisex excluded), plus each gender vs the rest so unisex is kept
df["women"] = df["Gender"].map({"women": 1, "men": 0})
for g in ["women", "men", "unisex"]:
    df[f"{g}_vs_rest"] = (df["Gender"] == g).astype(int)

dimensions = {
    "new_world": ("new world", "old world"),
    "global_south": ("global south", "global north"),
    "women": ("women", "men"),
    "women_vs_rest": ("women", "men + unisex"),
    "men_vs_rest": ("men", "women + unisex"),
    "unisex_vs_rest": ("unisex", "men + women"),
}

In [7]:
def log_odds(X, group, prior_occurrences=5):
    """Smoothed log-odds ratio of a feature appearing in group 1 vs group 0.

    Each group gets a prior of `prior_occurrences` pseudo-occurrences of the feature at
    its overall rate, so rare features shrink strongly toward zero and common ones barely move.
    Positive = more associated with group 1.
    """
    p = X.mean()  # prior from the full dataset, so items absent from a subset still get a prior
    mask = group.notna()
    X, g = X[mask], group[mask].astype(bool)

    n_a, n_b = g.sum(), (~g).sum()
    k_a, k_b = X[g].sum(), X[~g].sum()
    a_yes, a_no = prior_occurrences, prior_occurrences * (1 - p) / p

    yes_a, no_a = k_a + a_yes, n_a - k_a + a_no
    yes_b, no_b = k_b + a_yes, n_b - k_b + a_no
    delta = np.log(yes_a / no_a) - np.log(yes_b / no_b)
    se = np.sqrt(1 / yes_a + 1 / no_a + 1 / yes_b + 1 / no_b)

    return pd.DataFrame({
        "log_odds": delta,
        "odds_ratio": np.exp(delta),
        "z": delta / se,
        "pct_group1": k_a / n_a * 100,
        "pct_group0": k_b / n_b * 100,
        "count": k_a + k_b,
    }).sort_values("z", ascending=False)

results = {
    (feat, dim): log_odds(X, df[dim])
    for feat, X in features.items()
    for dim in dimensions
}

In [8]:
def show(feat, dim, n=10):
    r = results[(feat, dim)].round(2)
    g1, g0 = dimensions[dim]
    print(f"{feat} — most {g1}")
    display(r.head(n))
    print(f"{feat} — most {g0}")
    display(r.tail(n).iloc[::-1])

show("accords", "global_south")
show("accords", "unisex_vs_rest")

accords — most global south


,log_odds,odds_ratio,z,pct_group1,pct_group0,count
oud,0.90,2.46,11.84,7.84,3.27,933
amber,0.30,1.35,6.78,26.02,20.69,5147
warm spicy,0.28,1.32,6.62,30.05,24.54,6080
caramel,0.55,1.74,5.01,3.24,1.84,487
musky,0.21,1.24,4.40,19.09,15.97,3941
woody,0.16,1.17,4.06,55.43,51.57,12531
vanilla,0.16,1.18,3.52,21.61,18.95,4644
animalic,0.35,1.41,3.44,3.81,2.69,683
lavender,0.26,1.30,2.92,4.85,3.75,938
sweet,0.12,1.12,2.88,33.29,30.74,7479


accords — most global north


,log_odds,odds_ratio,z,pct_group1,pct_group0,count
floral,-0.49,0.61,-10.26,18.58,27.25,6283
fresh,-0.32,0.72,-5.30,10.36,13.83,3218
green,-0.31,0.73,-5.21,10.74,14.19,3304
citrus,-0.18,0.83,-4.57,35.75,40.01,9493
ozonic,-0.59,0.55,-4.34,1.70,3.15,713
iris,-0.47,0.63,-4.02,2.49,3.99,913
aquatic,-0.43,0.65,-3.99,2.90,4.47,1026
yellow floral,-0.47,0.63,-3.87,2.30,3.70,845
soft spicy,-0.45,0.63,-3.73,2.24,3.56,814
white floral,-0.17,0.84,-3.70,21.17,24.18,5722


accords — most unisex


,log_odds,odds_ratio,z,pct_group1,pct_group0,count
oud,1.88,6.53,24.79,9.02,1.46,933
amber,0.73,2.07,22.55,30.19,17.25,5147
warm spicy,0.58,1.78,18.75,32.98,21.64,6080
leather,0.95,2.58,16.55,9.02,3.67,1294
balsamic,1.20,3.33,16.08,5.99,1.85,763
smoky,1.21,3.36,12.81,3.74,1.11,470
animalic,0.92,2.52,11.93,4.78,1.92,683
tobacco,1.06,2.90,10.64,3.07,1.05,408
woody,0.27,1.31,9.84,56.70,49.90,12531
aromatic,0.22,1.25,7.64,35.04,30.11,7625


accords — most men + women


,log_odds,odds_ratio,z,pct_group1,pct_group0,count
floral,-0.85,0.43,-23.95,16.04,30.85,6283
fruity,-0.73,0.48,-22.01,19.32,33.27,6932
white floral,-0.67,0.51,-18.75,16.19,27.35,5722
fresh,-0.78,0.46,-16.56,7.96,15.92,3218
citrus,-0.37,0.69,-12.80,33.55,42.22,9493
sweet,-0.27,0.76,-8.81,27.24,32.89,7479
aquatic,-0.64,0.53,-8.18,2.68,5.01,1026
tropical,-0.50,0.61,-5.62,2.16,3.53,743
violet,-0.48,0.62,-5.45,2.14,3.46,731
ozonic,-0.45,0.64,-5.06,2.14,3.35,713


In [9]:
# One vector per item. Notes are pooled across top/middle/base, and a note and an accord
# naming the same material are merged into one item (present if it's either a note or an accord)
ACCORD_TO_NOTE = {"oud": "agarwood (oud)", "anis": "anise"}  # same material, different names; others match exactly

notes, accords = features["notes"], features["accords"]
same = {a: ACCORD_TO_NOTE.get(a, a) for a in accords.columns if ACCORD_TO_NOTE.get(a, a) in notes.columns}
both = pd.DataFrame({n: notes[n] | accords[a] for a, n in same.items()})
combined = pd.concat([notes.drop(columns=list(same.values())), accords.drop(columns=list(same)), both], axis=1)

item_type = pd.Series("ingredient", index=combined.columns)
item_type[accords.columns.drop(list(same))] = "accord"
item_type[both.columns] = "ingredient + accord"

# gender: + women / - men   world: + new world / - old world   south: + global south / - global north
# unisex is a separate, non-spatial measure: + unisex / - gendered (men + women)
VECTOR_DIMS = {"gender": "women", "world": "new_world", "south": "global_south", "unisex": "unisex_vs_rest"}

vectors = pd.concat({axis: log_odds(combined, df[dim])["log_odds"] for axis, dim in VECTOR_DIMS.items()}, axis=1)
vectors["type"] = item_type
vectors["count"] = combined.sum()

vectors[vectors["type"] == "ingredient + accord"].sort_values("gender").round(2)

,gender,world,south,unisex,type,count
lavender,-2.90,0.16,0.48,-0.22,ingredient + accord,2704
whiskey,-2.26,0.57,0.58,0.99,ingredient + accord,87
tobacco,-2.15,-0.37,0.30,1.02,ingredient + accord,779
leather,-1.88,-0.36,0.14,0.69,ingredient + accord,2333
camphor,-1.43,-0.08,-0.09,1.09,ingredient + accord,52
cannabis,-1.13,-0.10,-0.82,1.39,ingredient + accord,70
cinnamon,-1.11,-0.39,0.27,0.35,ingredient + accord,1679
agarwood (oud),-1.10,-0.69,0.84,1.91,ingredient + accord,1613
rum,-1.07,0.07,0.15,1.11,ingredient + accord,355
anise,-0.67,-0.27,-0.26,-0.32,ingredient + accord,409


In [10]:
# a = gender, b = new/old world, c = global south/north, u = unisex vs gendered; count = fragrances containing it
out = vectors.rename(columns={"gender": "a", "world": "b", "south": "c", "unisex": "u"})[["type", "a", "b", "c", "u", "count"]]
# Store odds ratios (exp of log-odds): 1 = no lean, >1 = toward the + end, <1 = toward the - end
out[["a", "b", "c", "u"]] = np.exp(out[["a", "b", "c", "u"]])
out.rename_axis("ingredient").reset_index().to_csv("ingredient_vectors.csv", index=False)
len(out)

749

In [11]:
import json

# One example fragrance per item for the page tooltip: the most-rated fragrance that contains the item
# (as an ingredient or an accord) and sits on the item's side of each axis:
#   gender: unisex if the item leans unisex (u > 1), otherwise women/men by the gender axis
#   world:  new/old world      south: global south/north
# If nothing matches exactly, any axis whose lean is under 1.1x ("tiny") may also match its other side.
# If a unisex-leaning item still has no unisex match, fall back to men's/women's by the gender axis.
TINY = np.log(1.1)

def pretty(slug):
    return slug.replace("-", " ").title()

def targets(v):
    gendered = "women" if v["gender"] > 0 else "men"
    return {
        "gender": "unisex" if v["unisex"] > 0 else gendered,
        "world": v["world"] > 0,
        "south": v["south"] > 0,
    }

def allowed(v, relax, gendered_fallback=False):
    t = targets(v)
    tiny = lambda axis: relax and abs(v[axis]) < TINY
    gendered = "women" if v["gender"] > 0 else "men"
    if gendered_fallback:
        gender = {"women", "men"} if tiny("gender") else {gendered}
    else:
        gender = {t["gender"]}
        if t["gender"] == "unisex" and tiny("unisex"):
            gender.add(gendered)
        elif t["gender"] != "unisex" and tiny("gender"):
            gender |= {"women", "men"}
    return {
        "gender": gender,
        "world": {True, False} if tiny("world") else {t["world"]},
        "south": {True, False} if tiny("south") else {t["south"]},
    }

def example_for(item, v):
    has_item = combined[item].astype(bool)
    passes = [(False, False), (True, False)]
    if targets(v)["gender"] == "unisex":
        passes.append((True, True))
    for relax, gendered_fallback in passes:
        a = allowed(v, relax, gendered_fallback)
        mask = (
            has_item
            & df["Gender"].isin(a["gender"])
            & df["new_world"].astype(bool).isin(a["world"])
            & df["global_south"].astype(bool).isin(a["south"])
        )
        if mask.any():
            break
    else:
        return None

    f = df.loc[df.loc[mask, "Rating Count"].idxmax()]
    t = targets(v)
    actual = {"gender": f["Gender"], "world": bool(f["new_world"]), "south": bool(f["global_south"])}
    return {
        "name": pretty(f["Perfume"]),
        "house": pretty(f["Brand"]),
        "year": None if pd.isna(f["Year"]) else int(f["Year"]),
        "country": f["Country"],
        "gender": f["Gender"],
        "world": "new world" if actual["world"] else "old world",
        "south": "global south" if actual["south"] else "global north",
        "off_side": [axis for axis in t if actual[axis] != t[axis]],  # axes it doesn't match (tiny lean or unisex fallback)
        "ratings": int(f["Rating Count"]),
        "top": f["Top"],
        "heart": f["Middle"],
        "base": f["Base"],
        "accords": [f[f"mainaccord{i}"] for i in range(1, 6) if pd.notna(f[f"mainaccord{i}"])],
    }

examples = {item: example_for(item, v) for item, v in vectors.iterrows()}
with open("ingredient_examples.json", "w", encoding="utf-8") as fh:
    json.dump(examples, fh, ensure_ascii=False)

found = {k: e for k, e in examples.items() if e}
no_match = [k for k, e in examples.items() if e is None]
print(f"{len(found)} with an example: {sum(e['gender'] == 'unisex' for e in found.values())} unisex, "
      f"{sum(bool(e['off_side']) for e in found.values())} not an exact match (tiny-lean or unisex fallback); {len(no_match)} without")
vectors.loc[no_match, ["type", "gender", "world", "south", "unisex", "count"]].round(2)

743 with an example: 395 unisex, 40 not an exact match (tiny-lean or unisex fallback); 6 without


,type,gender,world,south,unisex,count
tea rose,ingredient,1.10,-0.50,0.12,-0.20,38
white grapefruit,ingredient,-0.02,0.12,0.31,0.24,17
exotic spices,ingredient,-0.19,0.71,0.37,0.36,10
spruce,ingredient,-0.64,0.63,0.17,0.45,16
brandy,ingredient,-0.66,0.29,0.38,0.49,20
lemon zest,ingredient,-0.52,0.34,0.24,0.28,50


In [12]:
df.head()

,url,Perfume,Brand,Country,Gender,Rating Value,Rating Count,Year,Top,Middle,...,mainaccord2,mainaccord3,mainaccord4,mainaccord5,new_world,global_south,women,women_vs_rest,men_vs_rest,unisex_vs_rest
0,https://www.fragrantica.com/perfume/xerjoff/ac...,accento-overdose-pride-edition,xerjoff,Italy,unisex,1.42,201,2022.0,"fruity notes, aldehydes, green notes","bulgarian rose, egyptian jasmine, lily-of-the-...",...,woody,fruity,aromatic,floral,0,0,NaN,0,0,1
1,https://www.fragrantica.com/perfume/jean-paul-...,classique-pride-2024,jean-paul-gaultier,France,women,1.86,70,2024.0,"yuzu, citruses","orange blossom, neroli",...,white floral,sweet,fresh,musky,0,0,1.0,1,0,0
2,https://www.fragrantica.com/perfume/jean-paul-...,classique-pride-2023,jean-paul-gaultier,France,unisex,1.91,285,2023.0,"blood orange, yuzu","neroli, orange blossom",...,white floral,sweet,fresh spicy,musky,0,0,NaN,0,0,1
3,https://www.fragrantica.com/perfume/bruno-bana...,pride-edition-man,bruno-banani,Germany,men,1.92,59,2019.0,"guarana, grapefruit, red apple","walnut, lavender, guava",...,nutty,woody,tropical,NaN,0,0,0.0,0,1,0
4,https://www.fragrantica.com/perfume/jean-paul-...,le-male-pride-collector,jean-paul-gaultier,France,men,1.93,632,2020.0,"mint, lavender, cardamom, artemisia, bergamot","caraway, cinnamon, orange blossom",...,warm spicy,fresh spicy,cinnamon,vanilla,0,0,0.0,0,1,0
